# Solution — Fraud Detection Pipeline: Producer

Generates random banking transactions and streams them to the `transactions` Kafka topic.

> Run the consumer notebook first so messages are consumed as they arrive.

In [ ]:
from kafka import KafkaProducer
from datetime import datetime
from random import uniform, choice
from time import sleep
import json

# ----- Configuration -----
TOPIC   = 'transactions'
BROKERS = ['course-kafka:9092']

ACCOUNTS          = ['ACC_1', 'ACC_2', 'ACC_3', 'ACC_4', 'ACC_5']
CITIES            = ['Tel Aviv', 'Jerusalem', 'Haifa', 'Beer Sheva', 'Eilat']
CURRENCIES        = ['ILS', 'USD', 'EUR']
TRANSACTION_TYPES = ['TRANSFER', 'WITHDRAWAL', 'PURCHASE', 'DEPOSIT']
MERCHANTS         = ['Amazon', 'Zara', 'Apple', 'IKEA', 'Spotify', 'Netflix']

In [ ]:
def transaction_generator():
    """
    Infinite generator that yields one random transaction dict at a time.
    merchant is only populated for PURCHASE transactions — None otherwise.
    json.dumps handles Python None as JSON null automatically.
    """
    while True:
        tx_type = choice(TRANSACTION_TYPES)
        yield {
            'account_id':       choice(ACCOUNTS),
            'amount':           round(uniform(10, 8000), 2),
            'currency':         choice(CURRENCIES),
            'transaction_type': tx_type,
            'merchant':         choice(MERCHANTS) if tx_type == 'PURCHASE' else None,
            'city':             choice(CITIES),
            'timestamp':        datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

In [ ]:
# compression_type='gzip' reduces network overhead for JSON payloads.
# The consumer decompresses transparently — no changes needed on the consumer side.
#
# acks=1  → broker leader confirms receipt before we continue (balanced durability)
# retries=3 → automatically retry up to 3 times on transient failures
producer = KafkaProducer(
    bootstrap_servers = BROKERS,
    acks              = 1,
    retries           = 3,
    compression_type  = 'gzip'
)

In [ ]:
for tx in transaction_generator():

    # Kafka messages are always raw bytes — serialize the dict to UTF-8 JSON bytes.
    value = json.dumps(tx).encode('utf-8')

    # Use account_id as the key so all transactions from the same account
    # always land in the same partition → guaranteed ordering per account.
    # The key must also be bytes, not a plain string.
    key = tx['account_id'].encode('utf-8')

    producer.send(topic=TOPIC, key=key, value=value)

    # flush() blocks until the broker confirms receipt.
    # Without this, the message lives in an internal buffer and could be
    # lost if the process exits before the buffer is drained.
    producer.flush()

    merchant_str = f"@ {tx['merchant']}" if tx['merchant'] else ''
    print(f"Sent → {tx['account_id']} | {tx['amount']} {tx['currency']} | {tx['transaction_type']} {merchant_str}")

    sleep(uniform(1, 3))